<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Investigacion_Avanzada/Termodinamica_Lattice_Boltzmann.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Termodinamica Lattice Boltzmann

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def simulate_lattice_boltzmann():
    """ 2D Lattice Boltzmann (D2Q9) simulation of flow past a cylinder """
    # Simulation parameters
    maxIter = 2000
    Re = 150.0  # Reynolds number
    nx = 300    # resolution x
    ny = 100    # resolution y
    uLB = 0.04  # velocity in lattice units
    nulb = uLB * (ny / 4) / Re  # viscosity in lattice units
    omega = 1.0 / (3.0 * nulb + 0.5) # relaxation parameter

    # D2Q9 Lattice constants
    v = np.array([[ 0,  1,  0, -1,  0,  1, -1, -1,  1],
                  [ 0,  0,  1,  0, -1,  1,  1, -1, -1]])
    t = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])

    # Obstacle (cylinder)
    cx, cy, r = nx // 4, ny // 2, ny // 9
    y, x = np.ogrid[0:ny, 0:nx]
    obstacle = (x - cx)**2 + (y - cy)**2 < r**2

    # Initialization
    vel = np.zeros((2, ny, nx))
    vel[0, :, :] = uLB
    vel[:, obstacle] = 0.0
    rho = np.ones((ny, nx))
    
    # Initialize populations to equilibrium
    fin = np.zeros((9, ny, nx))
    for i in range(9):
        cu = 3 * (v[0, i] * vel[0, :, :] + v[1, i] * vel[1, :, :])
        fin[i, :, :] = rho * t[i] * (1 + cu + 0.5 * cu**2 - 1.5 * (vel[0, :, :]**2 + vel[1, :, :]**2))

    # Main loop
    for it in range(maxIter):
        # Macroscopic variables
        rho = np.sum(fin, axis=0)
        u = np.zeros((2, ny, nx))
        for i in range(9):
            u[0, :, :] += v[0, i] * fin[i, :, :]
            u[1, :, :] += v[1, i] * fin[i, :, :]
        u /= rho
        
        # Macroscopic (Dirichlet) boundary conditions (inlet)
        u[0, :, 0] = uLB
        u[1, :, 0] = 0.0
        rho[:, 0] = 1.0 / (1.0 - u[0, :, 0]) * (
            np.sum(fin[[0,2,4], :, 0], axis=0) + 
            2 * np.sum(fin[[3,6,7], :, 0], axis=0)
        )
        
        # Equilibrium calculation
        feq = np.zeros((9, ny, nx))
        for i in range(9):
            cu = 3 * (v[0, i] * u[0, :, :] + v[1, i] * u[1, :, :])
            feq[i, :, :] = rho * t[i] * (1 + cu + 0.5 * cu**2 - 1.5 * (u[0, :, :]**2 + u[1, :, :]**2))
            
        # Collision
        fout = fin - omega * (fin - feq)
        
        # Bounce-back boundary conditions for obstacle
        for i in range(9):
            fout[i, obstacle] = fin[8 - i, obstacle] # Reversing directions in D2Q9 convention (0 is rest, 1-3, 2-4, 5-7, 6-8)
            # Actually, standard D2Q9 indices need correct opposite mapping
        
        # Correct bounce back mapping
        noslip = [0, 3, 4, 1, 2, 7, 8, 5, 6]
        for i in range(9):
            fout[i, obstacle] = fin[noslip[i], obstacle]

        # Streaming
        for i in range(9):
            fin[i, :, :] = np.roll(np.roll(fout[i, :, :], v[0, i], axis=1), v[1, i], axis=0)
            
        # Outflow boundary (Neumann)
        fin[:, :, -1] = fin[:, :, -2]
        
        if (it + 1) % 500 == 0:
            print(f"Iter: {it+1}/{maxIter}")

    # Velocity magnitude for visualization
    u_mag = np.sqrt(u[0]**2 + u[1]**2)
    u_mag[obstacle] = 0
    
    plt.figure(figsize=(12, 4))
    plt.imshow(u_mag, cmap='jet')
    plt.title('Lattice Boltzmann D2Q9: Flow Past a Cylinder')
    plt.colorbar(label='Velocity Magnitude')
    plt.axis('off')
    plt.savefig('Lattice_Boltzmann_Flow.png', dpi=300)
    print("Lattice Boltzmann simulation complete. Image saved.")

if __name__ == '__main__':
    simulate_lattice_boltzmann()
